In [29]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('C:/Users/nebula/Desktop/ansys_use.fld', sep=r'\s+', comment='#')


filepath = "C:/Users/nebula/Desktop/ansys_use.fld"
with open(filepath, "r") as f:
    lines = f.readlines()
    print(lines[0:10])
    
    data_lines = lines[3:100]
    cleaned = "\n".join(
        re.sub(r"\bNan\b", "nan", line, flags=re.IGNORECASE) for line in data_lines
    )
    data = np.fromstring(cleaned, sep=" ")
    data = data.reshape(-1, 6)
    print(data[0:20])
    data[:, :3] = data[:, :3]*1000
    print(data[0:20])
    data[:, :3] = np.rint(data[:, :3])
    print(data[0:20])
    

C:\Users\nebula\AppData\Local\Temp\ipykernel_21712\1490578836.py:5: DtypeWarning: Columns (0: Grid, 1: Output, 2: Min:) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:/Users/nebula/Desktop/ansys_use.fld', sep=r'\s+', comment='#')


['Grid Output Min: [-52mm -111mm -52mm] Max: m 54mm 71mm] Grid Size: m 1mm 1mm] \n', 'X, Y, Z, Vector data "<Jx,Jy,Jz>"\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -5.2000000000000005e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -5.1000000000000004e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -5.0000000000000003e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.9000000000000002e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.8000000000000001e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.7000000000000007e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.6000000000000006e-02  Nan Nan Nan\n', '-5.2000000000000005e-02 -1.1100000000000000e-01 -4.5000000000000005e-02  Nan Nan Nan\n']
[[-0.052 -0.111 -0.051    nan    nan    nan]
 [-0.052 -0.111 -0.05     nan    nan    nan]
 [-0.052 -0.111 -0.049    nan    nan    nan]
 [-0.052 -0.111 -0.04

In [30]:
import re
import numpy as np
 
MU0 = 4 * np.pi * 1e-7  # H/m
 
 
# ---------------------------------------------------------------------
# 1. Parse the .fld file: header (grid bounds/spacing) + data rows
# ---------------------------------------------------------------------
def parse_fld(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()
    print('total lines in file=',len(lines))
    header1 = lines[0]  # "Grid Output Min: [...] Max: [...]"
    header2 = lines[1]  # "Grid Size: [...] Unit: "mm""
 
    nums = lambda s: [float(x) for x in re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)]
 
    min_max_vals = nums(header1)
    grid_min = np.array(min_max_vals[0:3])   # in mm
    grid_max = np.array(min_max_vals[3:6])   # in mm
 
    size_vals = nums(header2)
    grid_size = np.array(min_max_vals[6:9])     # spacing in mm

    print('line 1779117',lines[1779117])
 
    unit_match = re.search(r'Unit:\s*"(\w+)"', header1)
    unit = unit_match.group(1) if unit_match else "mm"
    print('unit ',unit)
    unit_scale = {"mm": 1e-3, "m": 1.0, "cm": 1e-2}.get(unit, 1e-3)

    # data starts after the 3rd line (header1, header2, column-label line)
    data_lines = lines[2:]
    # replace Nan/NaN with 'nan' so numpy can parse it
    cleaned = "\n".join(
        re.sub(r"\bNan\b", "nan", line, flags=re.IGNORECASE) for line in data_lines
    )
    data = np.fromstring(cleaned, sep=" ")
    print('data max = ',data.max())
    data = data.reshape(-1, 6)  # X, Y, Z, Jx, Jy, Jz
    print('data max = ',data.max())
    data[:, :3] = data[:, :3]*1000
    data[:, :3] = np.rint(data[:, :3])
    print('data max = ',data.max())
 
    nx = int(round((grid_max[0] - grid_min[0]) / grid_size[0])) + 1
    ny = int(round((grid_max[1] - grid_min[1]) / grid_size[1])) + 1
    nz = int(round((grid_max[2] - grid_min[2]) / grid_size[2])) + 1
    
    print(nx,ny,nz)
 
    expected = nx * ny * nz
    if data.shape[0] != expected:
        raise ValueError(
            f"Parsed {data.shape[0]} rows but expected {expected} "
            f"({nx} x {ny} x {nz}) -- check ordering/header parsing."
        )
    print('data max = ',data.max())
    J = data[:, 3:6].reshape(nx, ny, nz, 3, order='F')
    J = np.nan_to_num(J, nan=0.0)
    print('J max = ',J.max())
    J_flat = np.column_stack([
        J[...,0].ravel(order='F'),
        J[...,1].ravel(order='F'),
        J[...,2].ravel(order='F')
    ])
 
    Jx = J[..., 0]
    Jy = J[..., 1]
    Jz = J[..., 2]
 
    xs = (grid_min[0] + np.arange(nx) * grid_size[0]) * unit_scale
    ys = (grid_min[1] + np.arange(ny) * grid_size[1]) * unit_scale
    zs = (grid_min[2] + np.arange(nz) * grid_size[2]) * unit_scale
    spacing = grid_size * unit_scale  # (dx, dy, dz) in meters
 
    return (xs, ys, zs), tuple(spacing), (Jx, Jy, Jz)



In [31]:
print(J_flat[:20])
print(J_flat[-20:])
print(J_flat.min(), J_flat.max())

[[ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [-0.0001824  -0.00093603  0.00293228]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [-0.00056959 -0.00100295  0.00291438]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [-0.00124615 -0.00072925  0.00265905]
 [ 0.          0.          0.        ]]
[[ 0.          0.          0.        ]
 [ 0.00241182  0.00394379 -0.00251802]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.       

In [32]:
# ---------------------------------------------------------------------
# 2. Biot-Savart kernel K(r) = r / |r|^3 (double-sized for linear conv)
# ---------------------------------------------------------------------
def build_kernel(grid_shape, spacing):
    nx, ny, nz = grid_shape
    dx, dy, dz = spacing
 
    kx = (np.arange(2 * nx - 1) - (nx - 1)) * dx
    ky = (np.arange(2 * ny - 1) - (ny - 1)) * dy
    kz = (np.arange(2 * nz - 1) - (nz - 1)) * dz
 
    Kx, Ky, Kz = np.meshgrid(kx, ky, kz, indexing="ij")
    r2 = Kx**2 + Ky**2 + Kz**2
    r2[r2 == 0] = np.inf
    r3 = r2 * np.sqrt(r2)
 
    return Kx / r3, Ky / r3, Kz / r3
 
 
def fft_convolve_full(a, b):
    """Linear (non-circular) 3D convolution via zero-padded FFT."""
    out_shape = np.array(a.shape) + np.array(b.shape) - 1
    # pad each axis up to a fast FFT length (power of 2 here for simplicity)
    fshape = [int(2 ** np.ceil(np.log2(s))) for s in out_shape]
 
    A = np.fft.rfftn(a, fshape)
    B = np.fft.rfftn(b, fshape)
    conv = np.fft.irfftn(A * B, fshape)
 
    slices = tuple(slice(0, s) for s in out_shape)
    return conv[slices]

In [33]:
# ---------------------------------------------------------------------
# 3. Compute B from the 6 cross-product convolutions
# ---------------------------------------------------------------------
def compute_B(J_components, kernel, spacing, grid_shape):
    Jx, Jy, Jz = J_components
    Rx, Ry, Rz = kernel
    dx, dy, dz = spacing
    dV = dx * dy * dz
    nx, ny, nz = grid_shape
 
    JyKz = fft_convolve_full(Jy, Rz)
    JzKy = fft_convolve_full(Jz, Ry)
    JzKx = fft_convolve_full(Jz, Rx)
    JxKz = fft_convolve_full(Jx, Rz)
    JxKy = fft_convolve_full(Jx, Ry)
    JyKx = fft_convolve_full(Jy, Rx)
 
    Bx_full = JyKz - JzKy
    By_full = JzKx - JxKz
    Bz_full = JxKy - JyKx
 
    start = (nx - 1, ny - 1, nz - 1)
    sl = tuple(slice(start[i], start[i] + grid_shape[i]) for i in range(3))
 
    Bx = MU0 / (4 * np.pi) * Bx_full[sl] * dV
    By = MU0 / (4 * np.pi) * By_full[sl] * dV
    Bz = MU0 / (4 * np.pi) * Bz_full[sl] * dV
    return Bx, By, Bz

In [34]:
#print(Jx.min(), Jx.max())
#print(Jy.min(), Jy.max())
#print(Jz.min(), Jz.max())

J_flat = np.column_stack([Jx.ravel(), Jy.ravel(), Jz.ravel()])
print(J_flat[:20])
print(J_flat[-20:])
print(J_flat.min(), J_flat.max())

[[ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [-0.0001824  -0.00093603  0.00293228]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [-0.00056959 -0.00100295  0.00291438]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [-0.00124615 -0.00072925  0.00265905]
 [ 0.          0.          0.        ]]
[[ 0.          0.          0.        ]
 [ 0.00241182  0.00394379 -0.00251802]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.       

In [35]:
MU0 = 4 * np.pi * 1e-7

filepath = "ansys_use.fld"
(xs, ys, zs), spacing, (Jx, Jy, Jz) = parse_fld(filepath)
dx, dy, dz = spacing
dV = dx*dy*dz
nx, ny, nz = Jx.shape


Xg, Yg, Zg = np.meshgrid(xs, ys, zs, indexing="ij")
r_grid = np.column_stack([
    Xg.ravel(),
    Yg.ravel(),
    Zg.ravel()
])

J_flat = np.column_stack([
    Jx.ravel(),
    Jy.ravel(),
    Jz.ravel()
])

total lines in file= 3087602
line 1779117 3.4000000000000002e-02 -3.9999999999999994e-02 3.5000000000000003e-02  -2.5657783130450168e-02 2.3512267866146873e-02 -1.9431091947852354e-03

unit  mm
data max =  nan
data max =  nan
data max =  nan
150 166 124
data max =  nan
J max =  596.2138591062898


In [36]:
import numpy as np

sensors = np.array([
[-0.12274821, -0.00183155, -0.03129627],
[-0.11215786,  0.02520829, -0.00545477],
[-0.12313145, -0.00429692,  0.00615058],
[-0.13076667, -0.03270174, -0.02091326],
[-0.11790896,  0.01039029,  0.03717755],
[-0.10490351,  0.02770007,  0.06487468],
[-0.11199525, -0.00509726,  0.07783398],
[-0.12350638, -0.02102071,  0.04899444],
[-0.08788396,  0.06775472,  0.01257415],
[-0.09006716,  0.05849325,  0.05113152],
[-0.06861918,  0.06607289,  0.07749521],
[-0.1064291 ,  0.03956537,  0.02466959],
[-0.08359646,  0.03660279,  0.09203812],
[-0.05339208,  0.039579  ,  0.11235582],
[-0.05899789,  0.0084891 ,  0.12685355],
[-0.09033943,  0.00498935,  0.10590117],
[-0.0617153 ,  0.0901175 ,  0.00582221],
[-0.02932635,  0.09954632,  0.00247048],
[-0.02809624,  0.09946464,  0.0358955 ],
[-0.06405286,  0.08653245,  0.04058821],
[-0.03034079,  0.08853841,  0.06895606],
[ 0.00367348,  0.06525804,  0.09734701],
[-0.01804246,  0.04099204,  0.11962349],
[-0.03345242,  0.06775967,  0.09589401],
[-0.02232625,  0.01141135,  0.13510841],
[ 0.0142074 ,  0.00524382,  0.13319386],
[ 0.00921805, -0.02713565,  0.14229371],
[-0.02738077, -0.02137144,  0.14426383],
[ 0.00425832,  0.09888614,  0.00017799],
[ 0.0059212 ,  0.09824169,  0.0339851 ],
[ 0.03595065,  0.08848877, -0.00104368],
[ 0.06381388,  0.06879415, -0.00081312],
[ 0.06819433,  0.06415027,  0.03378742],
[ 0.03837067,  0.08818733,  0.03254194],
[ 0.00585072,  0.08745682,  0.06738245],
[ 0.0401662 ,  0.07663559,  0.06526112],
[ 0.03902138,  0.0554683 ,  0.09222566],
[ 0.01821201,  0.03502234,  0.11755754],
[ 0.05011443,  0.02214293,  0.10702483],
[ 0.07564602,  0.00975751,  0.08378628],
[ 0.07281752, -0.02264858,  0.09738725],
[ 0.04661726, -0.00902044,  0.12130297],
[ 0.08174775,  0.03889192,  0.00366398],
[ 0.09113311,  0.00634521,  0.01437323],
[ 0.08462474,  0.02900218,  0.04197674],
[ 0.06936394,  0.04285421,  0.07045078],
[ 0.08989055, -0.00509042,  0.05463008],
[ 0.09362292, -0.02528236,  0.02618546],
[ 0.08975441, -0.05713583,  0.03753855],
[ 0.08704141, -0.03862285,  0.06759002],
[ 0.08867079, -0.00893988, -0.01631609],
[ 0.08721018, -0.03665479, -0.0421078 ],
[ 0.08563403, -0.06904574, -0.03178729],
[ 0.09038905, -0.04028651, -0.00462945],
[-0.12815476, -0.03557867,  0.0168795 ],
[-0.12558063, -0.06678738,  0.02895143],
[-0.12503582, -0.09519864,  0.00297711],
[-0.13225093, -0.06401631, -0.00843104],
[-0.1216209 , -0.05279083,  0.05942304],
[-0.11114652, -0.03902769,  0.089003  ],
[-0.09861651, -0.07116378,  0.1002623 ],
[-0.11084618, -0.08299201,  0.06971022],
[-0.10971262, -0.12341802,  0.01228769],
[-0.11429998, -0.09697979,  0.03917574],
[-0.09503002, -0.12315346,  0.04744794],
[-0.08589609, -0.14598976,  0.018203  ],
[-0.09118913, -0.02893657,  0.11565058],
[-0.06214363, -0.02420152,  0.13531755],
[-0.03259713, -0.05429459,  0.14303203],
[-0.07263941, -0.06367108,  0.12806808],
[-0.07535417, -0.09691906,  0.10779562],
[-0.06381255, -0.12767658,  0.08218902],
[-0.0673704 , -0.14157141,  0.05130737],
[-0.09145328, -0.10905461,  0.07803291],
[-0.03914428, -0.08635089,  0.13105372],
[-0.00253662, -0.09266272,  0.12907834],
[-0.00970129, -0.11845507,  0.10839131],
[-0.04308379, -0.11281952,  0.11014073],
[-0.03145579, -0.13782869,  0.08316699],
[-0.0354892 , -0.15201982,  0.05191647],
[-0.02238041, -0.16639285,  0.01893607],
[-0.05597076, -0.16075986,  0.02060307],
[ 0.03980087, -0.04127972,  0.13003206],
[ 0.06367365, -0.05472233,  0.10761247],
[ 0.03614929, -0.08216833,  0.12231004],
[ 0.00313783, -0.06036493,  0.14106604],
[ 0.02567959, -0.11397534,  0.10250092],
[ 0.03392523, -0.13014541,  0.07164902],
[-0.00226854, -0.1525241 ,  0.04798899],
[ 0.00196506, -0.13883752,  0.07887159],
[ 0.07638498, -0.07046032,  0.07919594],
[ 0.07870949, -0.08644066,  0.04907491],
[ 0.05973431, -0.11165718,  0.06082784],
[ 0.05518547, -0.09722243,  0.09223863],
[ 0.02925171, -0.14415327,  0.04115789],
[ 0.05527954, -0.12562667,  0.03051185],
[ 0.039472  , -0.14865953,  0.00459929],
[ 0.01031247, -0.16223146,  0.01316117],
[ 0.08605505, -0.07168463,  0.00606334],
[ 0.07811841, -0.09947393, -0.01904594],
[ 0.0622934 , -0.12671016, -0.00656002],
[ 0.07473352, -0.10058516,  0.01877019]])

In [38]:
import numpy as np

MU0 = 4 * np.pi * 1e-7

def biot_savart_chunked(sensors, r_grid, J_flat, dV, chunk_size=200000):
    B = np.zeros((len(sensors), 3))

    N = len(r_grid)

    for i, r in enumerate(sensors):
        Bi = np.zeros(3)

        for start in range(0, N, chunk_size):
            end = min(start + chunk_size, N)

            R = r - r_grid[start:end]
            R2 = np.sum(R**2, axis=1)
            mask = R2>0

            R = R[mask]
            J = J_flat[start:end][mask]
            R2 = R2[mask]
            R3 = R2 * np.sqrt(R2)

            Bi += np.sum(np.cross(J, R) / R3[:, None], axis=0)

        B[i] = MU0 / (4*np.pi) * Bi * dV
    return B

In [39]:
#finding b at a specific point
B_sensors = biot_savart_chunked(sensors, r_grid, J_flat, dV)

i = 100 #any index
print("sensor location:", sensors[i])
print("B-field there:", B_sensors[i])
B_mag = np.linalg.norm(B_sensors, axis=1)
print(B_mag[i])

sensor location: [ 0.0622934  -0.12671016 -0.00656002]
B-field there: [-1.88617657e-10  3.60547751e-12  4.02539800e-11]
1.9289894524744147e-10


In [40]:
s_hat = np.array([
[-0.9405183 ,  0.33972421, -0.00598173],
[-0.90779147,  0.41404592, -0.0668684 ],
[-0.96756366,  0.2520959 ,  0.01387867],
[-0.9802677 ,  0.19007457,  0.05517969],
[-0.95890498,  0.27667649,  0.06168843],
[-0.85276002,  0.41674583,  0.31502115],
[-0.88851043,  0.24119569,  0.39032167],
[-0.98301735,  0.11193251,  0.14553258],
[-0.78693402,  0.61494354, -0.0502203 ],
[-0.75821936,  0.59905616,  0.25741127],
[-0.58339784,  0.63128992,  0.51108729],
[-0.89406612,  0.44786754,  0.00781988],
[-0.65453725,  0.48052437,  0.58381179],
[-0.35930336,  0.52608301,  0.7708088 ],
[-0.39288038,  0.36800906,  0.84263932],
[-0.68687734,  0.30419228,  0.66010565],
[-0.49009632,  0.86514611, -0.10579193],
[-0.13620186,  0.98075433, -0.14018814],
[-0.09801997,  0.98625779,  0.13311932],
[-0.5464298 ,  0.82134681,  0.16328439],
[-0.13046608,  0.87036919,  0.47476257],
[ 0.15432532,  0.69366958,  0.70363874],
[-0.0075529 ,  0.53647759,  0.84383285],
[-0.15435074,  0.73828119,  0.65657239],
[-0.03203458,  0.36567935,  0.93015423],
[ 0.26045552,  0.36579845,  0.89349072],
[ 0.22471391,  0.09488466,  0.96983656],
[-0.03873627,  0.13956734,  0.98942254],
[ 0.14651519,  0.98129415, -0.12474025],
[ 0.17266886,  0.97475651,  0.14163226],
[ 0.41142194,  0.89893826, -0.15091204],
[ 0.72297245,  0.6690476 , -0.17255115],
[ 0.798578  ,  0.59459091,  0.09401075],
[ 0.43009155,  0.89667342,  0.1051684 ],
[ 0.14920407,  0.87330423,  0.46392302],
[ 0.42632502,  0.78714998,  0.44587391],
[ 0.45269166,  0.63559224,  0.62527674],
[ 0.24751266,  0.51319668,  0.821816  ],
[ 0.58653629,  0.36644406,  0.72225477],
[ 0.82866081,  0.27222807,  0.48889839],
[ 0.8205217 ,  0.09022653,  0.56451402],
[ 0.57390987,  0.2066797 ,  0.79239516],
[ 0.92846108,  0.34641947, -0.13421482],
[ 0.98268218,  0.15747169, -0.09799708],
[ 0.93445093,  0.31357594,  0.16884176],
[ 0.77572873,  0.45266956,  0.4398211 ],
[ 0.96953249,  0.1094227 ,  0.21933072],
[ 0.99894438, -0.02311174, -0.04001496],
[ 0.97909117, -0.20266722,  0.01806541],
[ 0.95307931, -0.06944183,  0.29472027],
[ 0.98173673,  0.09427421, -0.16558826],
[ 0.99444181,  0.00625967, -0.1050187 ],
[ 0.98766956, -0.1500433 , -0.04477349],
[ 0.9920748 , -0.08079195, -0.09566679],
[-0.99446867,  0.08451626,  0.06209573],
[-0.97459004, -0.16695638,  0.14921854],
[-0.93318191, -0.27610573,  0.22998636],
[-0.9903575 , -0.01804144,  0.13729335],
[-0.96128242, -0.12866207,  0.24367489],
[-0.88622559, -0.01651706,  0.46293459],
[-0.78706648, -0.27196468,  0.55371879],
[-0.87530134, -0.35818968,  0.32477477],
[-0.81452707, -0.49714859,  0.2989445 ],
[-0.88821556, -0.39680149,  0.23169854],
[-0.69349759, -0.65355966,  0.30327766],
[-0.55765917, -0.74484093,  0.36649325],
[-0.70794278,  0.07692336,  0.70214396],
[-0.43677068,  0.14406963,  0.88799341],
[-0.10960628, -0.17654612,  0.97821707],
[-0.49187807, -0.25009383,  0.83394418],
[-0.51716455, -0.56411303,  0.64361869],
[-0.38703597, -0.77182801,  0.50447705],
[-0.43531295, -0.83095371,  0.34645293],
[-0.6719511 , -0.60436993,  0.42805552],
[-0.15628141, -0.47270367,  0.86727395],
[ 0.12253519, -0.51930812,  0.84579486],
[ 0.05433985, -0.73530375,  0.67563671],
[-0.22332333, -0.68755882,  0.69088038],
[-0.09495618, -0.84864383,  0.52043732],
[-0.14040034, -0.9325472 ,  0.33271884],
[ 0.03444357, -0.92921476,  0.36792191],
[-0.31400268, -0.86737562,  0.38602414],
[ 0.54692686, -0.020636  ,  0.83688578],
[ 0.76727323, -0.13628673,  0.62658227],
[ 0.51071354, -0.39803171,  0.76202736],
[ 0.14521993, -0.21971367,  0.96467318],
[ 0.36607277, -0.71342204,  0.59760597],
[ 0.49196225, -0.7979288 ,  0.34827118],
[ 0.17154697, -0.93353395,  0.31477442],
[ 0.19742267, -0.86947956,  0.45282763],
[ 0.87474347, -0.31196498,  0.37076274],
[ 0.89804187, -0.41917898,  0.13351993],
[ 0.75459085, -0.61477112,  0.2293485 ],
[ 0.70576914, -0.52536631,  0.47540743],
[ 0.46707246, -0.85024178,  0.24279222],
[ 0.72771547, -0.66988722,  0.14704022],
[ 0.61934189, -0.750998  ,  0.2289846 ],
[ 0.30174927, -0.89713117,  0.322854  ],
[ 0.96807822, -0.24745477, -0.04061748],
[ 0.93345188, -0.35632401,  0.04057185],
[ 0.8022008 , -0.57987278,  0.14229516],
[ 0.8757422 , -0.47987792,  0.05229448]])


In [46]:
a = B_sensors
b = s_hat
B_model = np.sum(a*b, axis=1)
print(B_model)

[ 1.32289244e-11  5.69025740e-12  4.89765359e-12  1.08690330e-11
 -4.13052033e-12 -4.37832021e-12 -3.72646144e-12 -3.12020550e-12
 -1.24880633e-12 -3.69657409e-12 -1.24396047e-12 -4.55752875e-12
 -1.70468716e-12  2.20163936e-12  1.64631626e-12 -1.68859986e-12
  2.18762880e-12  2.87699553e-12  3.97409425e-12 -1.26051159e-12
  5.90231503e-12  1.24809877e-11  8.26918597e-12  5.83669783e-12
  7.02677238e-12  1.21950113e-11  8.40073679e-12  6.05363458e-12
  3.61577044e-12  9.31068922e-12  5.46064581e-12  1.46196154e-11
  5.87756817e-11  1.96884078e-11  1.19498358e-11  2.31633886e-11
  2.33478747e-11  1.29017911e-11  2.47961673e-11  4.44463851e-11
  3.34753941e-11  2.02016133e-11  2.52138921e-10  4.33642568e-11
  9.21290297e-11  5.23343369e-11  6.74141859e-11  4.63477044e-11
  5.00209338e-11  5.40906783e-11 -1.56495829e-10 -5.39847437e-11
 -1.03988455e-11  2.91723304e-11  3.74896766e-12  2.84444839e-12
  7.54476231e-12  8.98314430e-12 -2.71584455e-12 -4.08757126e-12
 -5.30384979e-12 -4.29716